# Clase 8 — Unidad 8: Machine Learning en Biomedicina

**Curso:** Introducción al Análisis de Datos — 2026, Segundo Semestre
**Material de referencia:** *Python Essentials for Biomedical Data Analysis* — Capítulo 8, "Machine Learning in Biomedicine"

## Objetivos de aprendizaje
- Distinguir los tipos de aprendizaje automático: **supervisado**, **no supervisado**, y las tareas de **clasificación** y **regresión**.
- Ejecutar el **flujo completo de ML**: preprocesar → dividir → entrenar → predecir → evaluar.
- Entrenar clasificadores (**regresión logística** y **random forest**) con scikit-learn.
- **Evaluar** un modelo: matriz de confusión, *accuracy*, *precision*, *recall* y *F1*.
- Comprender el **sobreajuste** y usar **validación cruzada**.
- Aplicar **clustering K-means** como ejemplo de aprendizaje no supervisado.

## Contenidos de la clase
1. Conceptos básicos de machine learning
2. El flujo de trabajo de ML
3. Clasificación supervisada de principio a fin
4. Evaluación de modelos
5. Validación cruzada y sobreajuste
6. Aprendizaje no supervisado: clustering
7. Ejercicios de práctica

> Esta clase corresponde a la **Unidad 8** del libro guía. Simularemos datos de pacientes donde la **presencia de una enfermedad depende de las variables clínicas** (con ruido), para que el modelo aprenda un patrón real pero imperfecto, como en la práctica.

## Preparación del entorno

Usaremos **scikit-learn** (la librería estándar de ML en Python), además de pandas, numpy, matplotlib y seaborn.

In [ ]:
%pip install numpy pandas scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(8)
N = 400

edad = np.random.normal(55, 12, N)
imc = np.random.normal(28, 5, N)
glucosa = np.random.normal(105, 25, N)
presion = np.random.normal(130, 18, N)
biomarcador = np.random.normal(50, 15, N)

# La probabilidad de enfermedad depende de las variables clínicas (+ azar).
# Usamos un score con variables centradas para obtener clases balanceadas y aprendibles.
z = (0.9 * (edad - 55) / 12 + 1.0 * (imc - 28) / 5 + 0.8 * (glucosa - 105) / 25
     + 0.7 * (presion - 130) / 18 + 1.1 * (biomarcador - 50) / 15)
prob = 1 / (1 + np.exp(-z))
enfermedad = np.random.binomial(1, prob)

df = pd.DataFrame({
    "edad": edad.round(0),
    "imc": imc.round(1),
    "glucosa": glucosa.round(1),
    "presion_sistolica": presion.round(0),
    "biomarcador": biomarcador.round(1),
    "enfermedad": enfermedad,     # 0 = sano, 1 = enfermo (variable a predecir)
})

AZUL, NARANJO = "#0072B2", "#E69F00"
print("Dataset creado:", df.shape)
print("Distribución de clases (enfermedad):")
print(df["enfermedad"].value_counts())
df.head()

## 1. Conceptos básicos de machine learning

El **machine learning (ML)** permite que un algoritmo **aprenda patrones a partir de datos** para hacer predicciones, sin ser programado con reglas explícitas. Sus grandes categorías:

| Tipo | Datos | Objetivo | Ejemplo biomédico |
|---|---|---|---|
| **Supervisado** | Etiquetados (con respuesta conocida) | Predecir una etiqueta o valor | Predecir si un paciente tiene una enfermedad |
| **No supervisado** | Sin etiquetas | Descubrir grupos o estructura | Agrupar pacientes por perfil clínico |
| **Semi-supervisado** | Parcialmente etiquetados | Aprovechar pocos datos etiquetados | — |
| **Por refuerzo** | Interacción con un entorno | Maximizar una recompensa | — |

Dentro del **supervisado**, hay dos tareas:
- **Clasificación:** la respuesta es una **categoría** (ej. enfermo / sano).
- **Regresión:** la respuesta es un **número continuo** (ej. nivel de glucosa).

En esta clase resolveremos un problema de **clasificación**: predecir `enfermedad` (0/1).

## 2. El flujo de trabajo de ML

Todo proyecto de ML supervisado sigue los mismos pasos:

1. **Preparar los datos:** separar las **características** (features, `X`) de la **variable objetivo** (target, `y`).
2. **Dividir** en conjunto de **entrenamiento** y de **prueba** (train/test), para evaluar en datos que el modelo **no vio**.
3. **Preprocesar** (ej. escalar las variables).
4. **Entrenar** el modelo con los datos de entrenamiento (`fit`).
5. **Predecir** sobre el conjunto de prueba (`predict`).
6. **Evaluar** el desempeño con métricas apropiadas.

## 3. Clasificación supervisada de principio a fin

### 3.1 Separar características (X) y objetivo (y), y dividir en train/test

Usamos `train_test_split`. El parámetro `stratify=y` mantiene la misma proporción de clases en ambos conjuntos, y `random_state` hace el resultado reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("enfermedad", axis=1)   # características
y = df["enfermedad"]                # objetivo

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)

### 3.2 Escalar las variables

Muchos modelos funcionan mejor si las variables están en una escala similar. **Importante:** el escalador se **ajusta solo con los datos de entrenamiento** y luego se aplica a ambos, para no "filtrar" información del conjunto de prueba.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)   # ajusta y transforma con train
X_test_esc = scaler.transform(X_test)         # solo transforma test

print("Medias de X_train tras escalar (≈0):", X_train_esc.mean(axis=0).round(2))

### 3.3 Entrenar una regresión logística

La **regresión logística** es un clasificador simple y muy usado para resultados binarios. Se entrena con `fit()` y predice con `predict()`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

modelo_lr = LogisticRegression(max_iter=1000)
modelo_lr.fit(X_train_esc, y_train)          # entrenar

y_pred_lr = modelo_lr.predict(X_test_esc)    # predecir
print("Accuracy (regresión logística):", round(accuracy_score(y_test, y_pred_lr), 3))

## 4. Evaluación de modelos

### 4.1 Matriz de confusión

La **matriz de confusión** compara las etiquetas reales con las predichas:
- **VP** (verdaderos positivos) y **VN** (verdaderos negativos): aciertos.
- **FP** (falsos positivos) y **FN** (falsos negativos): errores.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Sano (0)", "Enfermo (1)"],
            yticklabels=["Sano (0)", "Enfermo (1)"])
plt.title("Matriz de confusión — Regresión logística")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

### 4.2 Métricas: accuracy, precision, recall y F1

A partir de la matriz de confusión se calculan métricas clave:

| Métrica | Qué mide | Fórmula |
|---|---|---|
| **Accuracy** | Proporción de aciertos totales | (VP+VN) / total |
| **Precision** | De lo predicho como positivo, cuánto era correcto | VP / (VP+FP) |
| **Recall (sensibilidad)** | De los positivos reales, cuántos detectó | VP / (VP+FN) |
| **F1** | Media armónica de precision y recall | 2·P·R / (P+R) |

> En medicina, el **recall** suele ser crítico: un falso negativo (no detectar una enfermedad presente) puede ser más grave que un falso positivo. `classification_report` resume todas las métricas.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_lr, target_names=["Sano", "Enfermo"]))

### 4.3 Otro modelo: Random Forest

**Random Forest** es un método de *ensemble* que combina muchos árboles de decisión. Suele lograr buen desempeño y permite estimar la **importancia de cada variable**.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train_esc, y_train)
y_pred_rf = modelo_rf.predict(X_test_esc)

print("Accuracy (random forest):", round(accuracy_score(y_test, y_pred_rf), 3))
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_rf, target_names=["Sano", "Enfermo"]))

In [ ]:
# Importancia de cada variable según el random forest
importancias = pd.Series(modelo_rf.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(7, 4))
plt.barh(importancias.index, importancias.values, color=AZUL)
plt.title("Importancia de las variables (Random Forest)")
plt.xlabel("Importancia")
plt.tight_layout()
plt.show()

## 5. Validación cruzada y sobreajuste

### 5.1 Sobreajuste (overfitting)

Un modelo **sobreajustado** memoriza el conjunto de entrenamiento pero **generaliza mal** a datos nuevos. Se detecta cuando el desempeño en entrenamiento es **mucho mayor** que en prueba.

In [ ]:
# Comparar accuracy en entrenamiento vs prueba
for nombre, modelo, Xtr, Xte in [
    ("Regresión logística", modelo_lr, X_train_esc, X_test_esc),
    ("Random Forest", modelo_rf, X_train_esc, X_test_esc),
]:
    acc_train = accuracy_score(y_train, modelo.predict(Xtr))
    acc_test = accuracy_score(y_test, modelo.predict(Xte))
    print(f"{nombre}: train={acc_train:.3f} | test={acc_test:.3f} | brecha={acc_train - acc_test:.3f}")

El Random Forest suele tener una **brecha** mayor (train ≫ test): es señal de sobreajuste. La regresión logística, más simple, generaliza de forma más pareja.

### 5.2 Validación cruzada (k-fold)

En lugar de una sola división train/test, la **validación cruzada** divide los datos en *k* partes, entrena con *k−1* y evalúa con la restante, repitiendo *k* veces. Da una estimación **más robusta** del desempeño. Usamos un `Pipeline` para escalar dentro de cada partición (evita filtrar información).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scores = cross_val_score(pipe, X, y, cv=5)   # 5-fold

print("Accuracy en cada partición:", scores.round(3))
print(f"Promedio: {scores.mean():.3f} (± {scores.std():.3f})")

## 6. Aprendizaje no supervisado: clustering K-means

Cuando **no** hay etiquetas, el ML no supervisado busca **estructura oculta**. El **K-means** agrupa los datos en *k* clusters según su similitud. Aquí agrupamos a los pacientes en 3 grupos usando solo sus variables clínicas (sin usar `enfermedad`), y visualizamos con PCA (reducción a 2 dimensiones).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Escalamos todas las características (el clustering es sensible a la escala)
X_esc = StandardScaler().fit_transform(X)

# K-means con 3 grupos
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_esc)

# Reducimos a 2 dimensiones con PCA solo para poder graficar
pca = PCA(n_components=2)
coords = pca.fit_transform(X_esc)

PALETA = ["#0072B2", "#E69F00", "#009E73"]
plt.figure(figsize=(7, 5))
for k in range(3):
    mask = clusters == k
    plt.scatter(coords[mask, 0], coords[mask, 1], s=25, alpha=0.7,
                color=PALETA[k], label=f"Cluster {k}")
plt.title("Clustering K-means de pacientes (proyección PCA)")
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Ejercicios de práctica

Usa el conjunto `df` (o crea tus propios datos) para resolver:

1. **Preparación:** separa `X` (características) e `y` (`enfermedad`) y divide en train/test (30% de prueba, `random_state=0`).
2. **Escalado:** ajusta un `StandardScaler` con el conjunto de entrenamiento y transforma ambos conjuntos.
3. **Entrenar:** entrena una **regresión logística** e imprime su *accuracy* en el conjunto de prueba.
4. **Matriz de confusión:** grafica la matriz de confusión del modelo anterior.
5. **Métricas:** imprime el `classification_report` e indica qué clase tiene mejor *recall*.
6. **Otro modelo:** entrena un `RandomForestClassifier` y compara su *accuracy* con la regresión logística.
7. **Validación cruzada:** calcula el *accuracy* promedio con validación cruzada de 5 particiones (usa un `Pipeline`).
8. **Desafío — clustering:** aplica K-means con `n_clusters=2` a las características escaladas y compara los grupos obtenidos con la variable real `enfermedad` (con una tabla de contingencia).

### Preguntas para reflexionar
1. ¿Por qué se evalúa el modelo con datos de prueba y no con los de entrenamiento?
2. ¿Qué diferencia hay entre aprendizaje supervisado y no supervisado?
3. En un examen para detectar una enfermedad grave, ¿te preocupa más la *precision* o el *recall*? ¿Por qué?
4. ¿Cómo se detecta el sobreajuste y por qué es un problema?
5. ¿Qué ventaja aporta la validación cruzada frente a una única división train/test?

In [ ]:
# Espacio para resolver los ejercicios
